# 07 - Model Evaluation

Comprehensive evaluation of trained XGBoost models including confusion matrices, ROC curves, and SHAP explanations.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path

TICKERS = ['NVDA', 'AMD', 'AAPL', 'TSLA', 'MSFT']
MODELS_DIR = Path('../models/saved')
DATA_DIR = Path('../data/processed')
LABEL_MAP = {0: 'Bearish', 1: 'Neutral', 2: 'Bullish'}

FEATURE_COLS = [
    'rsi_14', 'macd_signal', 'volume_ratio', 'price_momentum_5d',
]

In [ ]:
ticker = 'NVDA'
model_path = MODELS_DIR / f'xgb_{ticker}_latest.json'
model = xgb.XGBClassifier()
model.load_model(model_path)

df = pd.read_csv(DATA_DIR / f'{ticker}_final_dataset.csv')
available_features = [c for c in FEATURE_COLS if c in df.columns]
split = int(len(df) * 0.8)
X_test = df[available_features].iloc[split:].fillna(0)
y_test = df['direction_label'].iloc[split:]

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=list(LABEL_MAP.values())))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(LABEL_MAP.values()))
disp.plot(cmap='Blues')
plt.title(f'{ticker} - Confusion Matrix')
plt.show()

In [ ]:
# SHAP Feature Importance
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, class_names=list(LABEL_MAP.values()))